In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#LIBRARIES

#system
import os
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"
import sys

#math and array operations
import numpy as np
import math

#plotting
import matplotlib
# matplotlib.use("Agg") #UNCOMMENT IF PLOTTING WITHIN JUPYTER DOCUMENT
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import TwoSlopeNorm
from matplotlib.lines import Line2D


import cartopy.crs as ccrs
import cartopy.feature as cfeature

#data classes
import xarray as xr
import h5py
import pickle 

#loading bar
from tqdm import tqdm

#dates
from datetime import datetime

In [ ]:
#Importing DirectoryManager Class
sys.path.append(os.path.join("/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/","DataAnalysis"))
from CLASSES_Directories import DirectoryManager_Class

In [ ]:
DirectoryManager = DirectoryManager_Class()

codeType = os.path.join("DataAnalysis", "MPAS_Model_Data", "VariableComparisons")
# dataType = "AreaAverages"
dataType = "AreaAverages_Interpolation" #*INTERPOLATION

outputDirectory = DirectoryManager.GetOutputDirectory(codeType, dataType)
outputPlottingDirectory = DirectoryManager.GetOutputPlottingDirectory(codeType, dataType)

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_ModelData import StructuredModelData_Class, DataOperator_Class

In [ ]:
#Setup

Region = "TRACER"; Case = "WET"; spinup_hours = "0"
Region = "TRACER"; Case = "DIURNAL"; spinup_hours = "-5"

Region = "PRECIP"; Case = "WET"; spinup_hours = "12"
Region = "PRECIP"; Case = "DIURNAL"; spinup_hours = "12"

Region = "Hawaii"; Case = "WET"; spinup_hours = "12"
# Region = "Hawaii"; Case = "TRADES"; spinup_hours = "24" 

In [ ]:
#Load Model Directory Class
RunType = (Region,Case,"NSSL",spinup_hours)
ModelData_NSSL = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)

RunType = (Region,Case,"TEMPO",spinup_hours)
ModelData_TEMPO = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_DataSaving import DataSaving_Class

#Importing PlottingModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_PlottingModelData import RadarPlotting_Class

#Importing Radar Classes
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","Observation_Data"))
from CLASSES_RadarDataLoading import RadarData_MRMS_Class, RadarObservationMask_Class

sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_RadarDataPlotting import RadarPlotting_Class

#Importing PlottingModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_JobArray import JobArray_Class

#IMPORT FUNCTIONS
# --- Add your Functions folder to sys.path ---
import sys
path = os.path.join(DirectoryManager.mainCodeDirectory, 'Functions_2.0')
sys.path.append(path)


# --- Import all your function modules ---
import importlib
modules = [
    "AreaAverageFunctions",
    "ComputationFunctions",
    "DataFunctions",
    "DerivativeFunctions",
    "PlottingFunctions",
    "StatisticalFunctions",
]
for mod in modules:
    globals()[mod] = importlib.import_module(mod)        # import module itself
    globals().update(vars(globals()[mod]))              # import all functions into global namespace

In [ ]:
####################################
#DATA LOADING FUNCTIONS

In [ ]:
#Loading Radar Mask
RadarDataMask = RadarObservationMask_Class.LoadMaskData(DirectoryManager, ModelData_NSSL)

In [ ]:
#Data Loading Functions
def GetVariableSubset_Helper(ModelData, dataSubset, dataSubset_diag, dataSubset_static, varName):
    """
    Retrieves a variable subset from the given model data.
    If varName contains a '+', returns the sum of the two variables.
    """
    if '+' in varName:
        var1, var2 = varName.split('+')
        var1 = var1.strip()
        var2 = var2.strip()

        subset1 = DataOperator_Class.GetData_Variable(ModelData, dataSubset, 
                                                      dataSubset_diag, dataSubset_static, var1)
        subset2 = DataOperator_Class.GetData_Variable(ModelData, dataSubset, 
                                                      dataSubset_diag, dataSubset_static, var2)
        variableSubset = subset1 + subset2
    else:
        variableSubset = DataOperator_Class.GetData_Variable(ModelData, dataSubset, 
                                                             dataSubset_diag, dataSubset_static, varName)

    return variableSubset

def GetVariableSubset(ModelData, dataBundle, varName, interpolate=False, applyMask=False, cache=None):
    """
    dataBundle = (dataSubset, dataSubset_diag, dataSubset_static, zGrid_f, zGrid_c)
    interpolate=True reproduces the old behavior (3D fields interpolated to fixed MSL levels).
    cache is a dict that stores zTarget so GetZTarget runs once per RunCalculations call.
    """
    dataSubset, dataSubset_diag, dataSubset_static, zGrid_f, zGrid_c = dataBundle

    # Subsetting data
    variableSubset = GetVariableSubset_Helper(ModelData, dataSubset, dataSubset_diag,
                                              dataSubset_static, varName)

    # Interpolating Z levels  #*INTERPOLATION
    if interpolate and any(dim.startswith("nVertLevels") for dim in variableSubset.dims):
        if cache is None: cache = {}
        if "zTarget" not in cache:
            cache["zTarget"] = ModelData.GetZTarget(zGrid_f, zGrid_c)
        zTarget_f, zTarget_c = cache["zTarget"]
        variableSubset = ModelData.InterpolateVertical(variableSubset, zGrid_f, zGrid_c,
                                                       zTarget_f, zTarget_c)

    if varName in ['refl10cm', 'refl10cm_1km']:
        variableSubset = variableSubset.where(variableSubset > 0)

    # Applying RadarDataMask
    if applyMask:
        variableSubset = ApplyRadarMask(variableSubset)
    return variableSubset

def ApplyRadarMask(field):
    return field.where(RadarDataMask == True)

def GetWaterPath(ModelData, dataBundle, species):
    """Column-integrated water path (kg m^-2 = mm) of the listed species, on the native levels."""
    zGrid_f = dataBundle[3]
    rho = GetVariableSubset(ModelData, dataBundle, "rho", applyMask=False)
    vertDim = [d for d in rho.dims if d.startswith("nVertLevels")][0]
    dz = xr.DataArray(np.diff(np.asarray(zGrid_f), axis=0), dims=rho.dims, coords=rho.coords)   # m
    q = sum(GetVariableSubset(ModelData, dataBundle, s, applyMask=False) for s in species)
    return (rho * q * dz).sum(dim=vertDim)

def GetNeededVariable(ModelData, dataBundle, varName,
                      hydrometeors=["qc", "qi", "qs", "qg", "qr"], cache=None):
    WATERPATH_PREFIX = "WaterPath_"
    if varName.startswith(WATERPATH_PREFIX):
        key = varName[len(WATERPATH_PREFIX):]
        species = hydrometeors if key == "total" else [key]
        W = GetWaterPath(ModelData, dataBundle, species)
        return ApplyRadarMask(W)
    return GetVariableSubset(ModelData, dataBundle, varName, applyMask=True, cache=cache)

In [ ]:
####################################
#CALCULATION FUNCTIONS

In [ ]:
#Calculation Functions
def RunCalculations(ModelData, varNames, loop_elements, hydrometeors=["qc", "qi", "qs", "qg", "qr"]):
    outputDictionary, cache = {}, {}
    WP = "WaterPath_"

    # which species are needed
    wanted = set()
    for v in varNames:
        if v.startswith(WP):
            key = v[len(WP):]
            wanted.update(hydrometeors if key == "total" else [key])

    for count, t in enumerate(tqdm(loop_elements, desc="Processing timesteps")):
        [dS, dD, dSt, lat, lon, zGrid_f, zGrid_c, _, _] = DataOperator_Class.GetData_Subset(ModelData, t)
        dataBundle = (dS, dD, dSt, zGrid_f, zGrid_c)

        Wspec = {}
        if wanted:
            rho = GetVariableSubset(ModelData, dataBundle, "rho", interpolate=False).load()   # once
            vertDim = [d for d in rho.dims if d.startswith("nVertLevels")][0]
            dz = xr.DataArray(np.diff(np.asarray(zGrid_f), axis=0), dims=rho.dims, coords=rho.coords)
            rho_dz = rho * dz                                                                  # once
            for s in wanted:
                q = GetVariableSubset(ModelData, dataBundle, s, interpolate=False).load()
                Wspec[s] = (rho_dz * q).sum(dim=vertDim)

        for varName in varNames:
            if count == 0: print(f"Running for {varName}")
            if varName.startswith(WP):
                key = varName[len(WP):]
                field = sum(Wspec[s] for s in hydrometeors) if key == "total" else Wspec[key]
                field = ApplyRadarMask(field)                 # mask AFTER the vertical sum
            else:
                field = GetVariableSubset(ModelData, dataBundle, varName, applyMask=True, cache=cache)

            if count == 0:
                outputDictionary[varName] = InitiateMatrix(field, fill_nan=False)
            outputDictionary[varName][t] = GetMean(field)
    return outputDictionary

def InitiateMatrix(variableSubset, fill_nan=False):
    """
    Initializes an output matrix for a given variable subset.
    """
    if "nVertLevels" in variableSubset.dims:
        shape = (ModelData_NSSL.Ntime, ModelData_NSSL.Nzc)

    elif "nVertLevelsP1" in variableSubset.dims:
        shape = (ModelData_NSSL.Ntime, ModelData_NSSL.Nzf)

    else:  # 2D variable case
        shape = (ModelData_NSSL.Ntime, 1)

    fill_value = np.nan if fill_nan else 0
    output = np.full(shape, fill_value, dtype=float)

    return output

def GetMean(variableSubset):
    #(1/A) times integral of phi dA 
    #dA is [(R*cos(Lat)dLon)][RdLat] = R^2 cos(Lat)dLatdLon ==> weight is simply cos(Lat)
    weights = np.cos(np.deg2rad(variableSubset.latitude))
    variableMean = variableSubset.weighted(weights).mean(
        dim=("latitude", "longitude"),
        skipna=True
    )
    return variableMean

def GetTimeHours(ModelData):
    t = [datetime.strptime(s, "%Y-%m-%d_%H.%M.%S") for s in ModelData.timeStrings]
    return np.array([(x - t[0]).total_seconds() / 3600 for x in t])

def DepletionTimescale(out, hours, idx, minRate=0.05):
    h   = hours[idx]
    W   = out["WaterPath_total"][idx].ravel()          # mm, domain mean
    acc = out["rainnc"][idx].ravel()                   # mm, domain-mean accumulation
    R   = np.gradient(acc, h)                          # mm/h
    valid = R > minRate
    tau_t    = np.where(valid, W / R, np.nan)          # hours, time series
    tau_bulk = W[valid].mean() / R[valid].mean()       # hours, one number
    return h, W, R, tau_t, tau_bulk

def LoadOrRun_Depletion(ModelData_NSSL, ModelData_TEMPO, varNames, idx, overwrite=False):
    filePath = DataOperator_Class.GetOutputFilePath(
        ModelData_NSSL, DirectoryManager, outputDirectory,
        fileName=f"DepletionTimescale_{len(idx)}steps_{idx[0]}-{idx[-1]}.pkl")

    if os.path.exists(filePath) and not overwrite:
        with open(filePath, "rb") as f:
            saved = pickle.load(f)
        if saved["varNames"] == varNames and saved["idx"] == idx:
            print(f"Loaded {filePath}")
            return saved["out_NSSL"], saved["out_TEMPO"]
        print("Saved settings differ ==> recomputing")

    out_NSSL  = RunCalculations(ModelData_NSSL,  varNames, idx)
    out_TEMPO = RunCalculations(ModelData_TEMPO, varNames, idx)
    os.makedirs(os.path.dirname(filePath), exist_ok=True)
    with open(filePath, "wb") as f:
        pickle.dump({"varNames": varNames, "idx": idx,
                     "out_NSSL": out_NSSL, "out_TEMPO": out_TEMPO}, f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"Saved {filePath}")
    return out_NSSL, out_TEMPO

def ComputeDepletion(out, hours, idx, minRate=0.05, species=("qs", "qg")):
    h   = hours[idx]
    W   = out["WaterPath_total"][idx].ravel()
    acc = out["rainnc"][idx].ravel()
    R   = np.gradient(acc, h)
    valid = R > minRate

    result = {"hours": h, "W": W, "acc": acc, "R": R, "nValid": int(valid.sum()),
              "tau_t": np.where(valid, W / R, np.nan),
              "tau_bulk": W[valid].mean() / R[valid].mean() if valid.any() else np.nan}
    for s in species:
        Ws = out[f"WaterPath_{s}"][idx].ravel()
        result[f"W_{s}"] = Ws
        result[f"tau_{s}"] = Ws[valid].mean() / R[valid].mean() if valid.any() else np.nan   # share of tau
    return result

def MakeAllCalculations():
    varNames = ["WaterPath_total", "WaterPath_qs", "WaterPath_qg", "rainnc"]
    
    hours = GetTimeHours(ModelData_NSSL)
    idx = np.where((hours >= 12) & np.isclose(hours % 1, 0))[0].tolist()
    print(len(idx), "timesteps, about", round(len(idx) * 18 / 60), "min per scheme")
    
    out_NSSL, out_TEMPO = LoadOrRun_Depletion(ModelData_NSSL, ModelData_TEMPO, varNames, idx)
    
    results = {"NSSL":  ComputeDepletion(out_NSSL,  hours, idx),
               "TEMPO": ComputeDepletion(out_TEMPO, hours, idx)}
    return [results]

In [ ]:
####################################
#PLOTTING FUNCTIONS

In [ ]:
#Plotting Functions
def PlotDepletionTimeseries(results, minRate=0.05, title=None, fs=14):
    colors = {"NSSL": "blue", "TEMPO": "green"}
    fig, axes = plt.subplots(3, 1, figsize=(8, 9), sharex=True, gridspec_kw={"hspace": 0.12})

    for name, r in results.items():
        c = colors[name]
        axes[0].plot(r["hours"], r["W"],    color=c, lw=2.0, ls="-")
        axes[0].plot(r["hours"], r["W_qs"], color=c, lw=1.2, ls="--")
        axes[0].plot(r["hours"], r["W_qg"], color=c, lw=1.2, ls=":")
        axes[1].plot(r["hours"], r["R"], color=c, lw=2.0, label=name)
        axes[2].plot(r["hours"], r["tau_t"], color=c, lw=2.0, marker="o", ms=3,
                     label=f"{name}  (bulk {r['tau_bulk']:.1f} h)")
        axes[2].axhline(r["tau_bulk"], color=c, lw=0.8, ls="--")

    axes[1].axhline(minRate, color="gray", lw=0.6, ls=":")
    axes[0].set_ylabel("Water path (mm)", fontsize=fs)
    axes[1].set_ylabel("Resolved rain rate (mm h$^{-1}$)", fontsize=fs)
    axes[2].set_ylabel("Depletion time $\\tau$ (h)", fontsize=fs)
    axes[2].set_xlabel("Hours since start of simulation", fontsize=fs)

    styleKey = [Line2D([0], [0], color="k", lw=2.0, ls="-",  label="total"),
                Line2D([0], [0], color="k", lw=1.2, ls="--", label="snow"),
                Line2D([0], [0], color="k", lw=1.2, ls=":",  label="graupel")]
    axes[0].legend(handles=styleKey, frameon=False, fontsize=fs - 2, ncol=3)
    axes[1].legend(frameon=False, fontsize=fs - 2)
    axes[2].legend(frameon=False, fontsize=fs - 2)
    for ax in axes: ax.tick_params(labelsize=fs - 2)
    if title: axes[0].set_title(title, fontsize=fs + 2, fontweight="bold")
    return fig, axes

In [ ]:
####################################
#CALCULATING

In [ ]:
[results]=MakeAllCalculations()

In [ ]:
####################################
#PLOTTING

In [ ]:
fig, axes = PlotDepletionTimeseries(results, title=f"{Region} {Case}")